<a href="https://colab.research.google.com/github/GillValenzuela/curso_data_science/blob/master/DS_Ingemat_Clase_23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ---------------------------------------------------------------
# DEMO: One-Hot   vs.   Embedding  (PyTorch)
# ---------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

In [2]:
# 1. Vocabulario mínimo
vocab = {"<pad>": 0, "<unk>": 1, "i": 2, "love": 3, "like": 4, "pytorch": 5}
V = len(vocab)          # 6 tokens
d = 8                   # dimensión del embedding (pequeña para demo)

# 2. Frase de ejemplo
sentence = ["i", "love", "pytorch"]
idxs = torch.tensor([vocab[w] for w in sentence])          # [2, 3, 5]

# 3. ONE-HOT (sparse)
one_hot = F.one_hot(idxs, num_classes=V).float()           # (3, 6)

# 4. EMBEDDING (denso)
torch.manual_seed(0)                                       # reproducible
embed = nn.Embedding(num_embeddings=V, embedding_dim=d, padding_idx=0)
vecs = embed(idxs)                                         # (3, 8)

# 5. Similitud coseno entre "love" y "like"
love_vec = embed(torch.tensor([vocab["love"]]))            # (1, 8)
like_vec = embed(torch.tensor([vocab["like"]]))
cos_sim  = F.cosine_similarity(love_vec, like_vec).item()

# 6. Mostrar resultados
print("=== ONE-HOT (forma):", one_hot.shape)
print(one_hot, "\n")

print("=== EMBEDDING vectors (forma):", vecs.shape)
print(pd.DataFrame(vecs.detach().numpy(),
                   index=sentence, columns=[f"d{i}" for i in range(d)]), "\n")

print("Similitud coseno  <love, like>  =", round(cos_sim, 3))

=== ONE-HOT (forma): torch.Size([3, 6])
tensor([[0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 1.]]) 

=== EMBEDDING vectors (forma): torch.Size([3, 8])
               d0        d1        d2        d3        d4        d5        d6  \
i       -1.352654 -1.695931  0.566651  0.793508  0.598839 -1.555095 -0.341360   
love     0.750189 -0.585498 -0.173397  0.183478  1.389366  1.586334  0.946298   
pytorch -0.102310  0.792444 -0.289668  0.052507  0.522860  2.302205 -1.468894   

               d7  
i        1.853006  
love    -0.843677  
pytorch -1.586689   

Similitud coseno  <love, like>  = 0.209


In [3]:
!pip install torchtext scikit-learn tqdm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 59.0 MB/s eta 0:00:00


In [4]:
# ==============================================================
# BOW + MLP  •  IMDB sentiment  •  PyTorch + scikit-learn
# ==============================================================
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
from torchtext.datasets import IMDB

OSError: /usr/local/lib/python3.11/dist-packages/torchtext/lib/libtorchtext.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs

In [ ]:
# --------------------------------------------------------------
# 1) Cargar dataset IMDB (25 000 train + 25 000 test)
# --------------------------------------------------------------
train_iter = IMDB(split="train")
test_iter  = IMDB(split="test")


def load_IMDB(iterator):
    texts, labels = [], []
    label_map = {"neg": 0, "pos": 1}
    for label, text in iterator:
        texts.append(text)
        labels.append(label_map[label])
    return texts, labels

texts_train, labels_train = load_IMDB(train_iter)
texts_test,  labels_test  = load_IMDB(test_iter)


In [ ]:
# --------------------------------------------------------------
# 2) TF-IDF (1-gram + 2-gram)  •  max 20 000 features
# --------------------------------------------------------------
vectorizer = TfidfVectorizer(max_features=20_000,
                             ngram_range=(1, 2),
                             stop_words="english")

X_train = vectorizer.fit_transform(texts_train).astype("float32")
X_test  = vectorizer.transform(texts_test).astype("float32")

In [ ]:
 --------------------------------------------------------------
# 3) Convertir a tensores  + DataLoaders
# --------------------------------------------------------------
y_train = torch.tensor(labels_train, dtype=torch.long)
y_test  = torch.tensor(labels_test,  dtype=torch.long)

X_train_t = torch.tensor(X_train.toarray())
X_test_t  = torch.tensor(X_test.toarray())

train_ds = TensorDataset(X_train_t, y_train)
test_ds  = TensorDataset(X_test_t,  y_test)

train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=512)

In [ ]:
# --------------------------------------------------------------
# 4) Definir MLP
# --------------------------------------------------------------
input_dim = X_train_t.shape[1]     # = 20 000
model = nn.Sequential(
    nn.Linear(input_dim, 256),
    nn.ReLU(),
    nn.Linear(256, 2)              # 2 clases: neg / pos
)

In [ ]:
# --------------------------------------------------------------
# 5) Entrenar
# --------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

epochs = 3
for ep in range(1, epochs + 1):
    model.train()
    for xb, yb in tqdm(train_dl, desc=f"Epoch {ep}", leave=False):
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb)
        loss  = criterion(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # validación rápida
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for xb, yb in test_dl:
            preds = model(xb.to(device)).argmax(1).cpu()
            all_preds.extend(preds.numpy())
            all_true.extend(yb.numpy())
    acc = accuracy_score(all_true, all_preds)
    print(f"Epoch {ep}: test accuracy = {acc*100:.2f}%")

# --------------------------------------------------------------
# 6) Guardar modelo (opcional)
# --------------------------------------------------------------
torch.save(model.state_dict(), "bow_mlp_imdb.pt")